# Frontiere 2017: acoustic-wave measurement pilot

Section 4.1, Eqs. (94)–(95), Fig. 2 of [the CRKSPH paper](https://doi.org/10.1016/j.jcp.2016.12.004).
This notebook reads **local recordings**; it does not launch simulations or certify a paper reproduction.
The suite definition and outstanding setup checks are in `reproductions/frontiere2017/paper.yaml`.

The density reference and norm live in `warpSPHCases.reference.FrontiereAcousticReference` and have
manufactured-data tests. The norm gives each particle equal weight and evaluates the traveling wave
at the particle's **current** position and actual stored time. It is a linear reference at amplitude
$10^{-6}$; finite-amplitude Euler evolution is not asserted to be identical.


In [ ]:
# In an installed local environment, skip this cell. In Colab/Jupyter, install the current stack.
%pip install -q git+https://github.com/wi-re/warpSPHCases


Choose recording directories containing `trajectory.h5` **and** their resolved `caseSpec.json`.
Recordings can have been downloaded before opening this notebook. No HF connection is needed.
The sidecar must record `gamma`, `rho0`, `A`, `lamda`, and `c_s`; missing values are an error rather
than an invitation to assume defaults. Use output with a stored frame at $t=5$ for Fig. 2.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from warpSPHCases.store import Trajectory
from warpSPHCases.reference import buildReference

run_dirs = [
    # Path("/absolute/path/to/a/recording"),
]
target_time = 5.0
if not run_dirs:
    raise ValueError("Set run_dirs to one or more local recording directories, then rerun this cell.")


In [ ]:
rows = []
fig, axes = plt.subplots(len(run_dirs), 2, figsize=(10, 3 * len(run_dirs)), squeeze=False)
for directory, (profile_ax, error_ax) in zip(run_dirs, axes):
    with Trajectory(directory) as trajectory:
        reference = buildReference("frontiere2017-acoustic", trajectory)
        index = trajectory.frameAt(target_time)
        frame = trajectory.frame(index)
        x, measured = reference.measure(frame)
        exact = reference.evaluate(frame["t"], x)["density"]
        metrics = reference.metrics(trajectory, index)
        row = {"run": str(directory), "scheme": trajectory.scheme,
               "nx": trajectory.caseSpec.get("nx"), "particles": len(x),
               "time": frame["t"], "at_paper_time": bool(abs(frame["t"] - target_time) <= 1e-8),
               "density_dtype": str(frame["densities"].dtype), **metrics}
        rows.append(row)
        profile_ax.plot(x, (exact - 1) / 1e-6, label="linear reference")
        profile_ax.scatter(x, (measured["density"] - 1) / 1e-6, s=5, label=trajectory.scheme)
        profile_ax.set(xlabel="current x", ylabel="(rho - 1) / A", title=f"t={frame['t']:.9g}")
        profile_ax.legend()
        error_ax.plot(x, (measured["density"] - exact) / 1e-6, ".", ms=3)
        error_ax.axhline(0, color="0.6", lw=0.5)
        error_ax.ticklabel_format(axis="y", style="plain", useOffset=False)
        error_ax.set(xlabel="current x", ylabel="density error / A", title=str(directory.name))
fig.tight_layout()
for row in rows:
    print(row)


Before a convergence sweep, reconcile the paper protocol: B7 with support radius four particle
spacings, modified RK2, CFL 0.25, compatible energy, viscosity coefficients (2, 1), limiter and
adaptive support/volume rules. Verify initial sampling error and timestep handling. Merely selecting
`CRKSPH` does not establish these settings. Run one checked configuration before spending on a sweep.

The published counts are 20, 31, 50, 59, 126, 200, 316, 502, 796, 1261 and 2000. Preserve that list
when reproducing Fig. 2; a smaller demonstration has a different label. Float32 recordings can
show a quantization floor at this amplitude. Start with float64 and record the storage dtype.
Only fit comparable runs at the target time, with matching numerical protocol and a documented
fit interval. Report pairwise orders and any plateau; second order is a hypothesis to check,
not a preselected pass threshold. These measurements remain **unscored** until acceptance criteria
are supported by reference uncertainty or convergence evidence.
